In [1]:
import torch  # Import PyTorch library for tensor computations
import torch.nn as nn  # Import neural network modules
import torch.nn.functional as F  # Import functional API for activation and loss functions
from torch.utils.data import DataLoader, TensorDataset  # Import data loading utilities

# ----------------- Beta-VAE Model -----------------
class BetaVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=3, hidden_dim=32):
        super(BetaVAE, self).__init__()
        # Encoder layers
        self.fc1 = nn.Linear(input_dim, hidden_dim)  # Linear layer: input to hidden
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)  # Linear layer to compute mean of latent distribution
        self.fc_log_sigma2 = nn.Linear(hidden_dim, latent_dim)  # Linear layer to compute log variance of latent distribution
        # Decoder layers
        self.fc3 = nn.Linear(latent_dim, hidden_dim)  # Linear layer: latent to hidden
        self.fc4 = nn.Linear(hidden_dim, input_dim)  # Linear layer: hidden back to input dimension

    def encode(self, x):
        h1 = F.relu(self.fc1(x))  # Apply first layer followed by ReLU activation
        mu = self.fc_mu(h1)  # Compute latent mean
        log_sigma2 = self.fc_log_sigma2(h1)  # Compute log latent variance
        return mu, log_sigma2

    def reparameterize(self, mu, log_sigma2):
        std = torch.exp(0.5 * log_sigma2)  # Compute standard deviation from log variance
        eps = torch.randn_like(std)  # Sample epsilon from standard normal
        return mu + eps * std  # Return reparameterized latent vector z

    def decode(self, z):
        h3 = F.relu(self.fc3(z))  # Apply decoder's first layer and ReLU
        recon_x = self.fc4(h3)  # Reconstruct input from hidden representation
        return recon_x

    def forward(self, x):
        mu, log_sigma2 = self.encode(x)  # Encode input to latent parameters
        z = self.reparameterize(mu, log_sigma2)  # Sample latent vector via reparameterization
        recon_x = self.decode(z)  # Decode latent vector to reconstruction
        return recon_x, mu, log_sigma2

# ----------------- Beta-VAE Loss Function -----------------
def beta_vae_loss(recon_x, x, mu, log_sigma2, beta=4.0):
    recon_loss = F.mse_loss(recon_x, x, reduction='mean')  # Compute reconstruction loss (MSE)
    kld = -0.5 * torch.mean(1 + log_sigma2 - mu.pow(2) - log_sigma2.exp())  # Compute KL divergence to N(0, I)
    loss = recon_loss + beta * kld  # Total loss with beta weight on KL term
    return loss, recon_loss.detach(), kld.detach()

# ----------------- Example Usage -----------------
if __name__ == "__main__":
    X = torch.randn(372, 12)  # Generate synthetic data: 372 samples with 12 features
    dataset = TensorDataset(X)  # Wrap data in TensorDataset
    loader = DataLoader(dataset, batch_size=32, shuffle=True)  # DataLoader for training batches
    full_loader = DataLoader(dataset, batch_size=372)  # DataLoader for full dataset evaluation

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Select GPU if available, else CPU
    vae = BetaVAE(input_dim=12, latent_dim=3, hidden_dim=32).to(device)  # Initialize BetaVAE and move to device
    optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)  # Set up Adam optimizer

    beta_start, beta_max, warmup_steps = 0.0, 4.0, 50000  # Parameters for beta annealing schedule
    step = 0  # Initialize global step counter
    num_epochs = 100  # Define number of training epochs

    # Training loop
    for epoch in range(num_epochs):
        vae.train()  # Set model to training mode
        for batch in loader:
            X_batch = batch[0].to(device)  # Move batch data to device
            step += 1  # Increment step counter
            beta = min(beta_max, beta_start + (beta_max - beta_start) * step / warmup_steps)  # Compute current beta

            recon_batch, mu, log_sigma2 = vae(X_batch)  # Forward pass through VAE
            loss, recon_l, kld_l = beta_vae_loss(recon_batch, X_batch, mu, log_sigma2, beta)  # Compute losses

            optimizer.zero_grad()  # Reset gradients
            loss.backward()  # Backpropagate loss
            optimizer.step()  # Update model parameters

        print(f"Epoch {epoch+1}: Loss={loss.item():.4f}, Recon={recon_l:.4f}, KLD={kld_l:.4f}, Beta={beta:.2f}")  # Print training stats

    # Extract latent representations (mu)
    vae.eval()  # Set model to evaluation mode
    latents = []  # List to collect latent means
    with torch.no_grad():  # Disable gradient computation
        for batch in full_loader:
            X_full = batch[0].to(device)  # Move data to device
            _, mu_full, _ = vae(X_full)  # Get latent mean from VAE
            latents.append(mu_full.cpu())  # Move to CPU and append
    latents = torch.vstack(latents).numpy()  # Stack and convert to NumPy array
    print("Latent codes shape:", latents.shape)  # Display shape of latent codes


Epoch 1: Loss=0.9784, Recon=0.9784, KLD=0.0443, Beta=0.00
Epoch 2: Loss=0.9967, Recon=0.9966, KLD=0.0503, Beta=0.00
Epoch 3: Loss=0.7414, Recon=0.7413, KLD=0.0401, Beta=0.00
Epoch 4: Loss=0.8187, Recon=0.8185, KLD=0.0657, Beta=0.00
Epoch 5: Loss=0.8373, Recon=0.8369, KLD=0.0867, Beta=0.00
Epoch 6: Loss=0.8690, Recon=0.8686, KLD=0.0693, Beta=0.01
Epoch 7: Loss=1.0244, Recon=1.0238, KLD=0.0885, Beta=0.01
Epoch 8: Loss=0.9439, Recon=0.9432, KLD=0.0979, Beta=0.01
Epoch 9: Loss=0.7758, Recon=0.7745, KLD=0.1496, Beta=0.01
Epoch 10: Loss=1.0006, Recon=0.9989, KLD=0.1769, Beta=0.01
Epoch 11: Loss=0.9010, Recon=0.8993, KLD=0.1623, Beta=0.01
Epoch 12: Loss=0.8354, Recon=0.8317, KLD=0.3245, Beta=0.01
Epoch 13: Loss=0.8796, Recon=0.8745, KLD=0.4086, Beta=0.01
Epoch 14: Loss=0.9605, Recon=0.9547, KLD=0.4268, Beta=0.01
Epoch 15: Loss=0.9335, Recon=0.9245, KLD=0.6273, Beta=0.01
Epoch 16: Loss=0.7981, Recon=0.7841, KLD=0.9105, Beta=0.02
Epoch 17: Loss=0.7599, Recon=0.7475, KLD=0.7634, Beta=0.02
Epoch 